In [4]:
import chromadb

#1.创建客户端
client = chromadb.Client()

In [12]:
import chromadb
import numpy as np
from chromadb import EmbeddingFunction,Documents,Embeddings
from chromadb.api.types import Images
import torch

## 自定义模型

In [5]:
class MyEmbeddingFunction(EmbeddingFunction[Documents]):
    #初始化
    def __init__(self,len) -> None:
        self.len=len
        return
    #调用方法
    def __call__(self,input:Documents)->Embeddings:
        #返回长度为len的数组,元素都是input的长度
        return [ np.full(self.len,len(item)) for item in input]

In [14]:
#创建集合,指定嵌入函数
collection2=client.create_collection(name='collection2',embedding_function=MyEmbeddingFunction(10))
collection1=client.create_collection(name='collection1')

InternalError: Collection [collection2] already exists

In [22]:
print(collection2.configuration.get("embedding_function"))
print(collection2)
print(collection2._embedding_function)
#
print(collection1.configuration.get("embedding_function"))

None
Collection(name=collection2)


In [7]:
collection2.add(
    ids=['4','5','6'],
    documents=[
        "我闻琵琶已叹息",
        "凄凄惨惨戚戚",
        "断肠人在天涯",
    ]
)

In [8]:
collection2.peek()

{'ids': ['4', '5', '6'],
 'embeddings': array([[7., 7., 7., 7., 7., 7., 7., 7., 7., 7.],
        [6., 6., 6., 6., 6., 6., 6., 6., 6., 6.],
        [6., 6., 6., 6., 6., 6., 6., 6., 6., 6.]]),
 'documents': ['我闻琵琶已叹息', '凄凄惨惨戚戚', '断肠人在天涯'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [None, None, None]}

In [9]:
collection2.peek()['embeddings'].shape

(3, 10)

In [24]:
class ImageEmbeddingFunction(EmbeddingFunction[Images]):
    #初始化,传入自己的嵌入模型
    def __init__(self,model) -> None:
        self.model=model.to('cpu')
        return
    #调用方法
    def __call__(self,input:Images)->Embeddings:
        #将输入图像转换为Tensor
        input_tensor=torch.tensor(np.array(input),dtype=torch.float32)
        #前向传播,得到模型输出
        with torch.no_grad():
            output=self.model(input_tensor)
        #将处处转换为ndarray,返回
        return output.numpy()

In [25]:


# 假设你有一个预训练的模型（这里用一个简单的线性层代替）
class DummyModel(torch.nn.Module):
    def __init__(self):
        super(DummyModel, self).__init__()
        self.linear = torch.nn.Linear(3, 5)  # 输入3维，输出5维

    def forward(self, x):
        return self.linear(x)

# 测试数据：模拟图像输入（假设每张图像是一个3维向量）
test_images = [
    np.array([1.0, 2.0, 3.0]),
    np.array([4.0, 5.0, 6.0]),
    np.array([7.0, 8.0, 9.0])
]

# 创建自定义嵌入函数实例
model = DummyModel()
embedding_function = ImageEmbeddingFunction(model)

# 调用嵌入函数
embeddings = embedding_function(test_images)

# 打印结果
print("输入图像:")
for img in test_images:
    print(img)

print("\n生成的嵌入向量:")
for emb in embeddings:
    print(emb)


输入图像:
[1. 2. 3.]
[4. 5. 6.]
[7. 8. 9.]

生成的嵌入向量:
[ 2.3861227  -0.96795815  1.7379415  -0.07124379  0.39019918]
[ 5.301314   -0.84780794  3.2815495  -0.4692166   2.0746737 ]
[ 8.216505   -0.72765756  4.825157   -0.86718947  3.7591481 ]


## 第三方模型

In [ ]:
from chromadb.utils.embedding_functions import ChromaCloudQwenEmbeddingFunction

qwen_collection=client.get_or_create_collection(
    name='test_qwen',
    embedding_function=ChromaCloudQwenEmbeddingFunction
)